In [1]:
"""
BCT Analyzer Widget — extends ConnectivityMatrixEditor.

Usage
-----
from tvbwidgets.ui.connectivity_matrix_editor import ConnectivityMatrixEditor
from bct_analyzer_widget import BCTAnalyzerWidget

w = BCTAnalyzerWidget(connectivity=my_conn)
w.display()
"""

import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Import the existing editor — no code is duplicated
from tvbwidgets.ui.connectivity_matrix_editor_widget import ConnectivityMatrixEditor

from tvbwidgets import get_logger

LOGGER = get_logger(__name__)


# ---------------------------------------------------------------------------
# BCT metric catalogue  (pure NumPy, no extra dependency)
# ---------------------------------------------------------------------------

def _bct_strength(conn):
    return conn.weights.sum(axis=1)


def _bct_degree(conn):
    return (conn.weights > 0).sum(axis=1).astype(float)


def _bct_clustering_coeff(conn):
    W = conn.weights.copy()
    W /= W.max() if W.max() > 0 else 1.0
    W_13 = np.cbrt(W)
    cyc3 = np.diagonal(np.linalg.matrix_power(W_13, 3))
    K = (W > 0).sum(axis=1).astype(float)
    denom = K * (K - 1)
    return np.where(denom > 0, cyc3 / denom, 0.0)


def _bct_eigenvector_centrality(conn):
    W = conn.weights.copy()
    W /= W.max() if W.max() > 0 else 1.0
    eigvals, eigvecs = np.linalg.eigh(W)
    ec = np.abs(eigvecs[:, np.argmax(eigvals)])
    return ec / ec.max() if ec.max() > 0 else ec


def _bct_participation_coeff(conn):
    W = conn.weights.copy()
    n = W.shape[0]
    community = np.array([0] * (n // 2) + [1] * (n - n // 2))
    k = W.sum(axis=1)
    PC = np.ones(n)
    for m in np.unique(community):
        k_m = W[:, community == m].sum(axis=1)
        with np.errstate(invalid='ignore', divide='ignore'):
            PC -= np.where(k > 0, (k_m / k) ** 2, 0.0)
    return PC


BCT_METRICS = {
    "Strength (weighted degree)": {
        "description": "Sum of edge weights per node — high values indicate hub regions.",
        "compute": _bct_strength,
    },
    "Degree (binary)": {
        "description": "Number of non-zero connections per region (unweighted).",
        "compute": _bct_degree,
    },
    "Clustering Coefficient": {
        "description": "Weighted clustering coefficient (Onnela 2005) — local clique structure.",
        "compute": _bct_clustering_coeff,
    },
    "Eigenvector Centrality": {
        "description": "Influence score based on neighbour influence (leading eigenvector).",
        "compute": _bct_eigenvector_centrality,
    },
    "Participation Coefficient": {
        "description": "How evenly a node distributes connections across L/R hemispheres.",
        "compute": _bct_participation_coeff,
    },
}


# ---------------------------------------------------------------------------
# BCTAnalyzerWidget  — thin wrapper around ConnectivityMatrixEditor
# ---------------------------------------------------------------------------

class BCTAnalyzerWidget(ConnectivityMatrixEditor):
    """
    Extends ConnectivityMatrixEditor with:
      - A BCT metric dropdown (injected as a second row in the header).
      - A results panel below the matrix tab that recomputes automatically
        whenever the user saves or switches connectivity in history.
    """

    def __init__(self, connectivity, size=None, **kwargs):
        # Parent builds everything: header HBox, tab, canvases
        super().__init__(connectivity=connectivity, size=size, **kwargs)

        # BCT results panel (displayed below the tab)
        self.bct_output = widgets.Output(
            layout=widgets.Layout(border='1px solid #ccc', padding='8px', margin='4px')
        )

        # Inject the BCT row into the existing header
        self._add_bct_row()

        # First render of BCT results
        self._run_bct_metric()

    # ------------------------------------------------------------------
    # Inject BCT row into the parent's header
    # ------------------------------------------------------------------

    def _add_bct_row(self):
        self.bct_dropdown = widgets.Dropdown(
            options=list(BCT_METRICS.keys()),
            description='BCT Metric:',
            layout=widgets.Layout(width='360px'),
        )
        self.bct_dropdown.observe(self._on_bct_select, names=['value'])

        self.bct_desc_label = widgets.HTML(
            value=self._desc_html(self.bct_dropdown.value),
            layout=widgets.Layout(margin='0 0 0 10px'),
        )

        bct_row = widgets.HBox(
            [self.bct_dropdown, self.bct_desc_label],
            layout=widgets.Layout(align_items='center'),
        )

        # self.header was an HBox built by the parent — wrap both rows in a VBox
        self.header = widgets.VBox(
            [self.header, bct_row],
            layout=self.DEFAULT_BORDER,
        )

    def _desc_html(self, metric_name):
        desc = BCT_METRICS[metric_name]['description']
        return (
            "<span style='font-size:12px;color:#555;font-style:italic'>"
            f"&#8505;&#65039; {desc}</span>"
        )

    # ------------------------------------------------------------------
    # BCT metric logic
    # ------------------------------------------------------------------

    def _on_bct_select(self, change):
        self.bct_desc_label.value = self._desc_html(change['new'])
        self._run_bct_metric()

    def _run_bct_metric(self):
        metric = BCT_METRICS[self.bct_dropdown.value]
        self.bct_output.clear_output(wait=True)
        with self.bct_output:
            try:
                values = metric['compute'](self.connectivity)
                self._display_bct_results(self.bct_dropdown.value, values)
            except Exception as exc:
                LOGGER.error(f"BCT computation failed: {exc}")
                print(f"Could not compute '{self.bct_dropdown.value}': {exc}")

    def _display_bct_results(self, metric_name, values):
        labels = self.connectivity.region_labels
        v_min, v_max = values.min(), values.max()
        v_range = v_max - v_min if v_max != v_min else 1.0

        rows_html = ""
        for rank, idx in enumerate(np.argsort(values)[::-1]):
            pct = int((values[idx] - v_min) / v_range * 100)
            bar_color = f"hsl({120 - int(pct * 1.2)}, 70%, 45%)"
            bg = '#f9f9f9' if rank % 2 == 0 else '#fff'
            rows_html += (
                f"<tr style='background:{bg}'>"
                f"<td style='padding:3px 8px;color:#888;text-align:right'>{rank + 1}</td>"
                f"<td style='padding:3px 8px;font-family:monospace;font-size:12px'>"
                f"{labels[idx]}</td>"
                f"<td style='padding:3px 8px'>"
                f"<div style='background:{bar_color};width:{pct}%;min-width:2px;"
                f"height:12px;border-radius:2px;display:inline-block'></div>"
                f"<span style='font-size:11px;margin-left:6px'>{values[idx]:.4f}</span>"
                f"</td></tr>"
            )

        display(widgets.HTML(value=f"""
        <div style='font-family:sans-serif;max-height:300px;overflow-y:auto'>
          <h4 style='margin:4px 0 8px;color:#333'>
            {metric_name}
            <span style='font-weight:normal;font-size:12px;color:#666'>
              &mdash; connectivity {self.connectivity.gid.hex[:8]}
            </span>
          </h4>
          <table style='border-collapse:collapse;width:100%;font-size:13px'>
            <thead>
              <tr style='background:#eee'>
                <th style='padding:4px 8px'>#</th>
                <th style='padding:4px 8px;text-align:left'>Region</th>
                <th style='padding:4px 8px;text-align:left'>Value</th>
              </tr>
            </thead>
            <tbody>{rows_html}</tbody>
          </table>
        </div>
        """))

    # ------------------------------------------------------------------
    # Hook into parent's save + history so BCT auto-refreshes
    # ------------------------------------------------------------------

    def on_click_save(self, change):
        super().on_click_save(change)  # parent handles all connectivity logic
        self._run_bct_metric()         # then refresh BCT panel

    def _get_history_dropdown(self):
        dropdown = super()._get_history_dropdown()

        # Add an extra observer so BCT refreshes on history switch too
        dropdown.observe(lambda change: self._run_bct_metric(), 'value')
        return dropdown

    # ------------------------------------------------------------------
    # Display
    # ------------------------------------------------------------------

    def display(self):
        super().display()          # renders header + matrix tab from parent
        display(self.bct_output)   # BCT results panel below

19-03-2026 03:07:45 - INFO - tvbwidgets - Version: 2.3.2
